In [ ]:
# Importing Libraries
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.io import read_image
import torchvision.transforms.functional as F_t
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets.utils import download_url
import matplotlib.pyplot as plt
from IPython import display
import cv2
from PIL import Image
import numpy as np
import os


try:
    from einops import rearrange
except ImportError:
    %pip install einops
    from einops import rearrange

In [ ]:
class RFF(nn.Module): # Random Fourier Features
    def __init__(self, in_features, out_features, gamma): # gamma is the scaling factor
        super().__init__() # super() allows us to refer to the parent class without naming it explicitly
        self.B = torch.randn(in_features, out_features)*gamma # Randomly initialize the weights

    def forward(self, x): # Forward pass
        return torch.hstack([torch.sin(torch.matmul(2*math.pi*x, self.B)), torch.cos(torch.matmul(2*math.pi*x, self.B))]) # Compute the random features

In [ ]:
class RFF_SR(nn.Module): # Random Fourier Features for Super-Resolution
    def __init__(self, in_features, fourier_features, out_features, gamma): # gamma is the scaling factor
        super().__init__() # super() allows us to refer to the parent class without naming it explicitly
        self.net = nn.Sequential(RFF(in_features, fourier_features, gamma), nn.Linear(2*fourier_features, out_features))
        
    def forward(self, x): # Forward pass
      return self.net(x) # Compute the random features

In [ ]:
def create_coord_map(image_path): # Function to create the coordinate map
        image = read_image(image_path) # Read the image
        h, w = image.shape[1], image.shape[2] # Get the height and width of the image
        image = image.float()/255 # Normalize the image
        image = image.permute(1, 2, 0) # Change the dimensions of the image
        # print(image)

        h_coords = torch.linspace(0, 1, steps=h) # Create a tensor of linearly spaced values between 0 and 1
        w_coords = torch.linspace(0, 1, steps=w) # Create a tensor of linearly spaced values between 0 and 1
        grid = torch.stack(torch.meshgrid(h_coords, w_coords), dim=-1) # Create a grid of coordinates
        # print(h, w)

        return grid, image # Return the grid and the image

In [ ]:
download_url("https://s.w-x.co/util/image/w/as17-148-22727_lrg.jpg?v=at&w=400&h=400", ".", "earth.jpg") # Download the image

In [ ]:
img_init = cv2.imread("earth.jpg") # Read the image
img_init = cv2.cvtColor(img_init, cv2.COLOR_BGR2RGB) # Convert the image from BGR to RGB
plt.imshow(img_init)  # Display the image
print(img_init.shape)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Check if a GPU is available
img_path = "earth.jpg" # Path to the image
grid, image = create_coord_map(img_path) # Create the coordinate map
print(grid.shape, image.shape) # Print the shapes of the grid and the image
grid, image = grid.squeeze().to(device), image.squeeze().to(device) # Squeeze the grid and the image and move them to the device
train_coords, train_rgbs = grid[::2, ::2].reshape(-1, 2), image[::2, ::2].reshape(-1, 3) # Downsample the grid and the image
print(train_coords.shape, train_rgbs.shape) # Print the shapes of the downsampled grid and the image
test_coords, test_rgbs = grid.reshape(-1, 2), image.reshape(-1, 3) # Reshape the grid and the image
print(test_coords.shape, test_rgbs.shape) # Print the shapes of the grid and the image  
train_rgbs_3d = torch.reshape(train_rgbs, (200, 200, 3)) # Reshape the image


In [ ]:
train_rgbs_np = train_rgbs_3d.cpu().numpy()
print(train_rgbs_np.shape)
plt.imshow(train_rgbs_np)

In [ ]:
SuperReso = RFF_SR(2, 5000, 3, gamma=10).to(device) # Create the Super-Resolution model
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01) # Create an Adam optimizer
train_mse_arr = [] # Array to store the training MSE loss
train_psnr_arr = [] # Array to store the training PSNR
test_mse_arr = [] # Array to store the testing MSE loss
test_psnr_arr = [] # Array to store the testing PSNR

for iter in range(1, 101): # Iterate 100 times
    SuperReso.train() # Set the model to training mode
    optimizer.zero_grad() # Zero the gradients
    output = SuperReso(train_coords) # Get the output
    train_loss = F.mse_loss(output, train_rgbs) # Compute the MSE loss
    train_loss.backward() # Backpropagate the loss
    optimizer.step() # Update the weights
    train_mse_arr.append(train_loss.item()) # Append the training loss to the array
    train_psnr_arr.append(10*torch.log10(255*255/train_loss)) # Append the training PSNR to the array

    with torch.no_grad(): # No need to compute gradients
        prediction = SuperReso(test_coords) # Get the prediction
        test_loss = F.mse_loss(prediction, test_rgbs) # Compute the MSE loss
        test_psnr = 10*torch.log10(255*255/test_loss) # Compute the PSNR
        test_mse_arr.append(test_loss.item()) # Append the testing loss to the array
        test_psnr_arr.append(test_psnr.item()) # Append the testing PSNR to the array
        print(f"At iteration {iter}, the PSNR is {test_psnr.item():.6f} and MSE loss is {test_loss.item():.6f}") # Print the PSNR and the MSE loss
    if iter % 10 == 0: # Plot the prediction every 10 iterations
        fig, axes = plt.subplots(1, 2, figsize=(15, 5)) # Create a figure
        axes[0].imshow(prediction.reshape_as(image).cpu()) # Plot the prediction
        axes[0].set_title(f"Prediction Result in Iteration {iter}") # Set the title
        axes[1].imshow(image.cpu()) # Plot the original image
        axes[1].set_title(f"Original Image") # Set the title
        plt.show() # Show the plot

In [ ]:
plt.plot(train_mse_arr) # Plot the training MSE loss
plt.xlabel("Number of Iterations") # Set the x-axis label 
plt.ylabel("MSE Loss") # Set the y-axis label
plt.title("MSE Loss vs Number of Iterations during Training") # Set the title
plt.show() # Show the plot

In [ ]:
for i in range(len(train_psnr_arr)):
    train_psnr_arr[i] = train_psnr_arr[i].item()

plt.plot(train_psnr_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title("PSNR vs Number of Iterations during Training")
plt.show()

In [ ]:
plt.plot(test_mse_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("MSE Loss")
plt.title("MSE Loss vs Number of Iterations during Testing")
plt.show()

In [ ]:
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title("PSNR vs Number of Iterations during Testing")
plt.plot(test_psnr_arr)
plt.show()